# MMIBC v2 - Colab quickstart

Run the cells **top to bottom**. Runtime > Change runtime type > GPU for anything beyond the smoke test.

What lives where (nothing is lost when the runtime resets):

| Location | Content |
|---|---|
| Google Drive `MyDrive/MMIBC/raw`, `processed`, `splits`, `runs`, `figures`, `cache` | **raw downloads**, processed images, frozen splits and pair tables, checkpoints, predictions, metrics, figures, model weights |
| Colab local disk `/content` | a fast staged copy of the processed images, and temporary space while a download is unpacked (lost on reset, never the only copy) |

The data audit **stops the run on a real problem** (leakage, label mismatch, duplicate images across splits, ...). Read its message instead of working around it.

**Code comes from GitHub, data lives on Drive.** The cell below clones `main` into `/content/MMIBC` every session; do not run code from a copy of the repository kept on Drive (it goes stale and silently shadows fixes). Drive holds `raw`, `processed`, `splits`, `runs`, `figures` only.

In [ ]:
import os
REPO = "/content/MMIBC"
if not os.path.exists(REPO):
    !git clone https://github.com/dev-tyta/MMIBC.git {REPO}
else:                                                    # an older runtime may still be on the old revision/v2 branch: switch to main, then update
    !git -C {REPO} fetch -q origin
    !git -C {REPO} checkout -q main
    !git -C {REPO} pull --ff-only
%cd {REPO}
!pip -q install -e ".[download,dicom]"

In [ ]:
import sys, os

REPO = "/content/MMIBC"
for path in [REPO, os.path.join(REPO, "src")]:      # the kernel started before `pip install -e` finished: make the package importable now
    if os.path.exists(path) and path not in sys.path:
        sys.path.insert(0, path)

import torch
print("CUDA:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available (CPU only)")
from mmibc.utils.paths import resolve_paths
paths = resolve_paths()          # mounts Google Drive and creates the folder layout
print(paths)

## 1. Smoke test (about a minute, CPU is fine)
Runs the **whole** pipeline on synthetic images with a tiny random backbone: prepare, audit, both task types, evaluation with controls, explanations, figures. If this fails, fix it before downloading anything.

In [ ]:
!mmibc smoke

## 2. Credentials
Add these as Colab secrets (key icon, left bar) and switch **Notebook access ON** for each one:

* `KAGGLE_API_TOKEN` - dataset downloads. Create it in your Kaggle account Settings with **Generate New Token** (what current `kagglehub` expects). The older pair `KAGGLE_USERNAME` + `KAGGLE_KEY` ("Create Legacy API Key") also works.
* `HF_TOKEN` - **only** for DINOv3 (gated: accept the licence on the model page first)

Colab secrets can only be read by notebook cells, **not** by the `!mmibc ...` commands below (separate processes). The next cell copies them into environment variables, which those commands inherit. Re-run it after adding or changing a secret. Values are never printed.

In [ ]:
from mmibc.data.download import export_colab_secrets
export_colab_secrets()   # prints one status line per secret, never the value
# Need at least one Kaggle credential ("exported"/"already set") before section 3. HF_TOKEN may stay missing: the DINOv3 rows of `suite backbones` are then SKIPPED (not failed) and run once you add it (section 6).

## 3. Data
Raw downloads are saved **directly to Google Drive** (`MyDrive/MMIBC/raw/<dataset>`), so they survive a runtime reset and are never downloaded twice. Before copying, the tool checks that your Drive has room and refuses to start otherwise; a download that was interrupted is detected and redone, never half-trusted.

`mmibc download` with no argument lists every registered dataset. The Kaggle slugs for VinDr come from the old code; the BUSI slug is an **unverified mirror** - `inspect` shows what actually arrived, and the audit checks it.

In [ ]:
!mmibc download

In [ ]:
from mmibc.data.download import export_colab_secrets
export_colab_secrets()   # Kaggle secrets must be copied into the environment by a notebook cell, in this runtime, before `!mmibc download`
!mmibc download vindr_images vindr_annotations busi

**Everything at once.** `--all` downloads every registered dataset one after the other, keeps going if one fails, and ends with a summary. Datasets that already arrived are skipped, so after fixing a failure you can simply run it again. Check your free Drive space first: VinDr-Mammo alone is large.

In [ ]:
from mmibc.data.download import export_colab_secrets
export_colab_secrets()   # Kaggle secrets must be copied into the environment by a notebook cell, in this runtime, before `!mmibc download`
!mmibc download --all

In [ ]:
!mmibc inspect vindr_annotations
!mmibc inspect busi

`bus_bra` downloads from Zenodo; `bus_uclm` from Kaggle (its layout is unverified, so you may need a `source_manifest.csv`). `breast_lesions_usg` downloads from TCIA as **two files**: the images-and-masks zip and the clinical spreadsheet.

For `bus_bra` and `breast_lesions_usg` the labels live in a spreadsheet, so `prepare us` **builds `raw/<name>/source_manifest.csv` by itself** the first time (BUS-BRA: `bus_data.csv`, with `Case` as patient id so the two breasts of a patient never straddle splits; BrEaST: the clinical `.xlsx`). `!mmibc manifest <name>` does just that step, `--force` rebuilds it; an existing manifest is never overwritten. A source listed as `data.us_external_source` is kept out of training entirely and used as an external test set (section 5C).

`cmmd` is a **mammography** set with biopsy-confirmed labels (not ultrasound). `prepare cmmd` reads its TCIA DICOM download - the `CMMD/` folder of `.dcm` files plus `CMMD_clinicaldata_revision.xlsx` - from `MyDrive/MMIBC/raw/cmmd/` (a nested folder such as `cmmd/TheChineseMammographyDatabase/` is found automatically); see section 5E. The Kaggle mirror in the registry has an unverified layout.

In [ ]:
from mmibc.data.download import export_colab_secrets
export_colab_secrets()   # Kaggle secrets must be copied into the environment by a notebook cell, in this runtime, before `!mmibc download`
!mmibc download bus_bra breast_lesions_usg
# !mmibc inspect bus_bra                # structure report
# !mmibc manifest bus_bra               # optional: `prepare us` does this when the manifest is missing
# !mmibc manifest breast_lesions_usg
OVER = "data.us_sources=[busi,bus_bra]"
POOL = "run_name=pool_busi_busbra"     # a different pool is a different experiment: give it its own run folder

## 4. Prepare and audit
`prepare` is resumable (re-run it after an interruption). `audit` prints every check - schema, leakage, near-duplicates across splits, label mapping, source-label confounding, pair integrity - and trains nothing.

In [ ]:
!mmibc prepare vindr -c vindr_multiview
!mmibc prepare us -c mammo_us_synthetic --set {OVER}      # builds raw/bus_bra/source_manifest.csv first if it is missing

In [ ]:
!mmibc audit -c vindr_multiview
!mmibc audit -c mammo_us_synthetic --set {OVER}

## 4b. Everything at once
**One command for all datasets.** `mmibc experiments` works out which datasets you have downloaded, prepares them (resumable), then runs every experiment of the paper in turn: native CC-MLO fusion and mammogram + ultrasound pairing with the controls on **VinDr-Mammo** and on **CMMD**, the robust recipe, and the external ultrasound test on BrEaST. A track whose dataset is not downloaded is **skipped with the reason**, a track that fails is recorded and the rest carry on; call the same command again after downloading or fixing something and only the missing tracks train. Sections 5-6 below are the same experiments one at a time.

`--list` first: it prints what would run, what is already finished and what is blocked (it prepares and trains nothing). VinDr (BI-RADS labels) and CMMD (biopsy labels) are never pooled into one training set - their numbers are separate results.


In [ ]:
from mmibc.data.download import export_colab_secrets
export_colab_secrets()       # makes HF_TOKEN (if you added it) visible to the !mmibc below; harmless otherwise
!mmibc experiments --list

In [ ]:
!mmibc experiments
# !mmibc experiments --suites          # also the fusion ladders and the backbone comparison (many more runs)
# !mmibc experiments --only cmmd       # one group (vindr | robust | cmmd | external | suites) or named tracks
# !mmibc experiments --xai             # also the explanation metrics for every run
# !mmibc experiments --set training.epochs=1 --only vindr   # a quick end-to-end check before the long run

In [ ]:
import pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 40)
summary = pd.read_csv(paths.runs / "experiments_all" / "summary.csv")
summary[[c for c in ("track", "state", "auroc", "auroc_lo", "auroc_hi", "ext_auroc", "auroc_D1", "auroc_D2", "minutes") if c in summary.columns]]
# the full table with the caveats: runs/experiments_all/summary.md ; per-track tracebacks: FAILED_<track>.txt next to it

## 5. Experiments
**A. Native pairs (CC and MLO of the same breast, VinDr-Mammo)** - validates the cross-attention fusion on genuine correspondence.

In [ ]:
!mmibc run -c vindr_multiview

**B. Synthetic mammogram + ultrasound pairs and their controls.** `label_matched` is the paper's setting. `random` and `shuffled_partner` are the controls a reviewer will ask for: if they score the same, the fusion is not using the pairing.

In [ ]:
for mode in ("label_matched", "random", "shuffled_partner"):
    !mmibc run -c mammo_us_synthetic --set task.pair_mode={mode} {OVER} {POOL}

**C. External ultrasound test (BrEaST-Lesions-USG).** The 256 BrEaST cases are held out *entirely* (never in training or validation) and paired with the held-out VinDr mammograms; the result is reported as `ext_test` next to the internal test. Needs `breast_lesions_usg` from section 3.

In [ ]:
!mmibc prepare us -c mammo_us_external      # adds BrEaST to the processed ultrasound pool (listed last: the BUSI/BUS-BRA splits do not change)
!mmibc audit -c mammo_us_external
!mmibc run -c mammo_us_external --set run_name=pool_ext

**D. Mitigations for the weak spots of the first real runs.** None of these has been validated on real data yet: they are hypotheses to test with the suites below, on identical splits.

| Problem seen | Mitigation (config / option) |
|---|---|
| noisy validation, best epoch 4 of 12 | `training.ema_decay=0.998`, `training.layer_decay=0.8`, `model.drop_path=0.1`, `training.label_smoothing=0.05` |
| probabilities worse than the prevalence (Brier skill < 0) | `eval.calibration=platt` (default now, fitted on validation only; AUROC unchanged) and the `spec90`/`sens90` operating points in `metrics.json` |
| mammogram explanations at chance (a lesion is ~1 patch at 224 px) | rectangular, higher-resolution input: `input.image_size_a=[448,266]` (`vindr_multiview_robust`, `mammo_us_robust`) |
| 0.93 AUROC not attributable to either stream | `pool_ladder` (both unimodal baselines) + `fusion_gain.csv`; `training.modality_dropout=0.25` |

`recipe` trains the baseline and each mitigation on the same VinDr split; then repeat the winner over seeds on the *same* split and read the mean ± sd and the seed-ensemble AUROC (single-seed differences of 0.01-0.03 are inside the confidence intervals).

In [ ]:
!mmibc suite recipe

In [ ]:
# after looking at results.md of suite_recipe: repeat the best entries over seeds (same partition, different training seeds)
# !mmibc suite recipe --only baseline robust --seeds 42 43 44
# !mmibc suite pool_ladder --seeds 42 43 44
# !mmibc run -c vindr_multiview_robust --set run_name=vindr_robust
# !mmibc run -c mammo_us_robust --set run_name=pool_robust

**E. CMMD - biopsy-labelled mammograms.** CMMD labels are *pathology* (benign / malignant), unlike VinDr's BI-RADS, so a mammogram + ultrasound pair that is "label matched" now means the same thing on both sides. `prepare cmmd` turns the DICOMs into PNGs (about 16 GB are read from Drive: the first run is slow, and it is resumable), labels **only the biopsied breast** of each study (the other breast of a four-image study has no recorded finding and is left out), and splits **by patient** (CMMD has no official split).

**Caution - D1 vs D2.** CMMD's two subsets differ hugely in malignant rate (D1 ~51 %, D2 ~98 % of breasts). A model can score well by recognising the subset. The audit warns (`subset_shortcut`), `metrics.json` holds AUROC per subset (`by_stratum`), and `data.cmmd.subsets=[D1]` removes the shortcut. Report the D1-only number as the headline. CMMD has no lesion boxes: localisation metrics are skipped for it and `size_conditioned` pairing is refused.

In [ ]:
!pip -q install pydicom openpyxl      # already included by the first cell; harmless to repeat
!mmibc prepare cmmd -c cmmd_multiview
!mmibc audit -c cmmd_multiview         # read the subset_shortcut / metadata_shortcut warnings before trusting any number
!mmibc run -c cmmd_multiview --set data.cmmd.subsets=[D1] run_name=cmmd_mv_d1     # shortcut-free protocol
# !mmibc run -c cmmd_multiview --set run_name=cmmd_mv_all                          # D1 + D2: compare through by_stratum
# !mmibc suite cmmd_pool_ladder --set data.cmmd.subsets=[D1]                       # CMMD x ultrasound pool: does the second modality add anything?

In [ ]:
import json
m = json.load(open(paths.runs / "cmmd_mv_d1_dinov2_base" / "metrics.json"))["splits"]["test"]
print("test AUROC", round(m["auroc"], 3), m.get("auroc_delong_ci"))
print(json.dumps(m.get("by_stratum"), indent=1))      # AUROC per CMMD subset (null = the subset has a single class)

## 6. Ablations on identical frozen test pairs
`ladder` = fusion variants, `pool_ladder` = the same on the mammogram + ultrasound pool **including both unimodal baselines** (without them the fused AUROC cannot be attributed to either stream), `backbones` = DINOv2 / DINOv2+registers / DINOv3 / CNNs, `pairing` = pairing tiers, `recipe` = the training mitigations (section 5D). Each prints one table with DeLong and McNemar tests against the reference run; suites with both unimodal runs also write `fusion_gain.csv` (paired ΔAUROC against each unimodal model and the best one chosen on validation).

Suites are **resumable**: if the runtime dies, run the same cell again. Finished runs are skipped, `suite_<name>/status.csv` shows what is done, `--force` redoes everything.

In [ ]:
!mmibc suite ladder
!mmibc suite pool_ladder
# !mmibc suite pairing

**Backbone comparison.** `backbones` trains the same cross-attention model on six backbones (DINOv2, DINOv2 + registers, DINOv3 ViT-B/16 and ViT-L/16, ConvNeXt-B, EfficientNet-B5) and compares them on identical frozen test pairs. The two DINOv3 rows are **gated**: until a Hugging Face token is visible they are `SKIPPED` (the other four run normally); after you accept the licence and add `HF_TOKEN` under Colab Secrets, run the same cell again and only those two rows train. `backbones_cmmd` is the same comparison on CMMD (biopsy labels). Differences of 0.01-0.03 AUROC are inside the confidence intervals of these test sets: read the DeLong / McNemar columns, not the ranking.

In [ ]:
from mmibc.data.download import export_colab_secrets
export_colab_secrets()      # copies HF_TOKEN (if you have added it) into the environment for the !mmibc below; harmless before
!mmibc suite backbones                                       # VinDr-Mammo, BI-RADS labels
!mmibc suite backbones_cmmd --set data.cmmd.subsets=[D1]     # CMMD, biopsy labels, subset-shortcut free (run section 5E's prepare first)
# the tables: runs/suite_backbones/results.md and runs/suite_backbones_cmmd/results.md ; status.csv shows done / SKIPPED / FAILED per row

## 7. Comparison with published results
`protocol_petrini` reproduces the labelling and split of the closest published baseline, so that comparison row is meaningful. Every other row is flagged *not comparable* - read the caveat column before quoting it.

In [ ]:
!mmibc run -c protocol_petrini
import glob
run = sorted(glob.glob(str(paths.runs / "protocol_petrini*")))[-1]
!mmibc literature {run} --dataset VinDr

## 8. Figures for the paper

In [ ]:
!mmibc figures --template lncs          # ROC, ladder, controls, calibration, explanation summary, data composition -> <root>/figures

In [ ]:
# Architecture and algorithm diagrams (TikZ). Pre-built copies are in paper/figures/pdf; rebuild only if you change them.
!apt-get -qq install -y texlive-latex-extra texlive-fonts-recommended poppler-utils > /dev/null
!mmibc figures --tex
from IPython.display import Image, display
display(Image("paper/figures/pdf/fig_architecture.png", width=900))

## 9. Real mammogram + ultrasound pairs (KAU-BCMD) - needs one manual step
The corpus reports 205 ultrasound cases that correspond to part of the mammogram cases, but the layout of the Kaggle mirror is not known. So: download, **inspect**, then write the pair manifest yourself.

In [ ]:
from mmibc.data.download import export_colab_secrets
export_colab_secrets()   # Kaggle secrets must be copied into the environment by a notebook cell, in this runtime, before `!mmibc download`
!mmibc download kau_bcmd
!mmibc inspect kau_bcmd

Write `pairs_manifest.csv` with columns `pair_id, case_id, path_a, path_b, label` (`path_a` = mammogram, `path_b` = ultrasound, paths relative to two root folders, `label` 0/1 from the corpus labels, **one row per patient pairing**). Downscale the images first (long side 672) - full-size PNGs are slow. Then evaluate a trained synthetic-pair model on real pairs (Tier 1b, evaluation only):

In [ ]:
# !mmibc evaluate -c mammo_us_synthetic --set task.pair_mode=label_matched {OVER} {POOL} \
#       task.real_pairs_csv=/content/kau/pairs_manifest.csv task.real_root_a=/content/kau task.real_root_b=/content/kau

## Troubleshooting
* **`audit` fails** - the message names the check and the rows; fix the data, do not raise the tolerance.
* **Out of GPU memory** - `--set training.batch_size=8 model.freeze=last_n`, or a smaller backbone (`model.backbone=dinov2_small`).
* **Runtime disconnected during training or a suite** - run the same command again; training resumes from the last atomic checkpoint (including the random-number state, and the EMA when used) and finished suite runs are skipped.
* **DINOv3 rows show `SKIPPED` in `suite_backbones/status.csv`** - no Hugging Face token was visible. Accept the licence on the DINOv3 model pages, add `HF_TOKEN` under Colab Secrets (Notebook access on), run the credentials cell (`export_colab_secrets()`), then run the same `!mmibc suite backbones` cell again: finished rows are kept, only the skipped ones train. A `401/403` after that means the licence was not accepted for that model. The default backbone `dinov2_base` needs neither.
* **`mmibc experiments` shows `SKIPPED` tracks** - the dataset named in the state is not downloaded (`!mmibc download <name>`; CMMD and BrEaST are separate downloads). Nothing is wrong: run the same command again and only the skipped tracks train. `FAILED` means the dataset is there but preparing or training raised: the traceback is in `runs/experiments_all/FAILED_<track>.txt`.
* **CMMD: `no clinical spreadsheet` / `download is incomplete` / `pydicom`** - `prepare cmmd` needs `CMMD_clinicaldata_revision.xlsx` next to the DICOM folder (the labels are not in the DICOM files), refuses a download that lacks more than 5 % of the files listed in TCIA's `metadata.csv` (delete `metadata.csv` only if you really use part of CMMD), and needs `pip install pydicom`.
* **Drive quota / "Not enough free space"** - raw downloads are saved on Drive by default (`paths.raw_on_drive=true`). If your Drive is too small, add `--set paths.raw_on_drive=false` to `download`, `inspect` and `prepare` to keep them on Colab's temporary disk instead (they are then re-downloaded after a reset).
* **"Google Drive is not mounted"** - run `from google.colab import drive; drive.mount('/content/drive')` in a cell (cell 2 does this); `!mmibc` commands cannot mount it themselves.
* **A host refuses the download (e.g. `HTTP 403` from Zenodo)** - the error now shows what the server answered. Download the file in your browser and upload it to `MyDrive/MMIBC/raw/<dataset>/` (for BUS-BRA: `BUSBRA.zip` into `raw/bus_bra/`), then run `!mmibc download <dataset>` again: it unpacks the zip you placed there and checks it against the published MD5 where one is known, without touching the network.
* **Kaggle datasets all fail with "credentials not visible"** - the cell that exports your secrets was not run in this runtime (secrets are copied into the environment by a notebook cell; `!mmibc` cannot read them itself). Run the credentials cell, or the download cell that now includes it, again.